# 07 Lazy Feature Sanity Check

This notebook checks the Stage 2 lazy feature pipeline in two layers:

- sequence-level lazy loading and alignment
- frame-level indexing with context stacking

The goal is to verify the data path once, then validate the two dataset views that use it.

In [6]:
from pathlib import Path
import importlib
import random
import sys

import torch

def find_project_root() -> Path:
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "src").exists() and (parent / "data").exists():
            return parent
    raise RuntimeError("Could not detect project root. Ensure 'src' and 'data' directories exist.")

project_root = find_project_root()
print(f"Detected project root: {project_root}")

stage2_dir = project_root / "src" / "07_lazy_feature_mlp"
for path in (project_root, stage2_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import lazy_dataset
import lazy_frame_dataset

importlib.reload(lazy_dataset)
importlib.reload(lazy_frame_dataset)

VADLazySequenceDataset = lazy_dataset.VADLazySequenceDataset
VADLazyFrameDataset = lazy_frame_dataset.VADLazyFrameDataset

generated_dir = project_root / "data" / "generated"
norm_stats_path = generated_dir / "train" / "features" / "noisy_norm_stats.npz"

print(f"Generated dir: {generated_dir}")
print(f"Norm stats path: {norm_stats_path}")
print(f"Norm stats exists: {norm_stats_path.exists()}")

Detected project root: /home/jasmi/Noise-Robust-Voice-Activity-Detection-FINAL
Generated dir: /home/jasmi/Noise-Robust-Voice-Activity-Detection-FINAL/data/generated
Norm stats path: /home/jasmi/Noise-Robust-Voice-Activity-Detection-FINAL/data/generated/train/features/noisy_norm_stats.npz
Norm stats exists: True


## 1. Sequence-Level Sanity Check

This checks the lazy sequence dataset before any context stacking is applied. Each item should represent one full example with frame-wise features and labels.

In [7]:
sequence_dataset = VADLazySequenceDataset(
    generated_dir=str(generated_dir),
    split="dev",
    manifest_type="noisy",
    norm_stats_path=str(norm_stats_path),
    subset_fraction=None,
)

print(f"Sequence dataset length: {len(sequence_dataset)}")

if len(sequence_dataset) == 0:
    raise RuntimeError("Sequence dataset is empty - check data paths and manifest files")

sample = sequence_dataset[0]
print(f"Sample ex_id: {sample['ex_id']}")
print(f"x shape: {sample['x'].shape}")
print(f"y shape: {sample['y'].shape}")
print(f"num_frames: {sample['num_frames']}")

assert sample['x'].ndim == 2, f"Expected [T, D] features, got {sample['x'].shape}"
assert sample['x'].shape[1] == 121, f"Expected 121 features, got {sample['x'].shape[1]}"
assert sample['y'].ndim == 1, f"Expected 1D labels, got {sample['y'].shape}"
assert sample['x'].shape[0] == sample['y'].shape[0], f"x frames {sample['x'].shape[0]} != y frames {sample['y'].shape[0]}"
assert sample['num_frames'] == sample['x'].shape[0], f"num_frames {sample['num_frames']} != x.shape[0] {sample['x'].shape[0]}"

unique_labels = sorted(sample['y'].unique().tolist())
print(f"Unique label values: {unique_labels}")
assert set(unique_labels).issubset({0.0, 1.0}), f"Unexpected labels: {unique_labels}"

x_np = sample['x'].numpy()
print(f"Feature mean: {x_np.mean():.4f}")
print(f"Feature std: {x_np.std():.4f}")
print(f"Feature min: {x_np.min():.4f}")
print(f"Feature max: {x_np.max():.4f}")
print("✓ Sequence-level checks passed")

Loaded 500 examples from dev noisy manifest
Sequence dataset length: 500
Sample ex_id: dev_0000000
x shape: torch.Size([1127, 121])
y shape: torch.Size([1127])
num_frames: 1127
Unique label values: [0.0, 1.0]
Feature mean: -0.0759
Feature std: 1.2583
Feature min: -9.5535
Feature max: 7.7805
✓ Sequence-level checks passed


## 2. Frame-Level Sanity Check

This checks the frame dataset wrapper, which turns each sequence into frame-level samples with context stacking.

In [8]:
frame_dataset = VADLazyFrameDataset(
    generated_dir=str(generated_dir),
    split="dev",
    manifest_type="noisy",
    norm_stats_path=str(norm_stats_path),
    subset_fraction=0.25,
    subset_seed=1337,
)

print(f"Frame dataset length: {len(frame_dataset)}")

if len(frame_dataset) == 0:
    raise RuntimeError("Frame dataset is empty - check data paths and manifest files")

sample = frame_dataset[0]
print(f"Sample ex_id: {sample['ex_id']}")
print(f"x shape: {sample['x'].shape}")
print(f"y shape: {sample['y'].shape}")
print(f"frame_idx: {sample['frame_idx']}")

assert sample['x'].shape == (1331,), f"Expected x shape (1331,), got {sample['x'].shape}"
assert sample['y'].dim() == 0, f"Expected scalar y, got shape {sample['y'].shape}"
assert 0.0 <= sample['y'].item() <= 1.0, f"Unexpected label value: {sample['y'].item()}"
print("✓ Frame-level sample checks passed")

Loaded 500 examples from dev noisy manifest
Applied subset sampling: 125/500 examples (25.00%)
Frame dataset: 280928 total frames from 125 sequences
Frame dataset length: 280928
Sample ex_id: dev_0000002
x shape: torch.Size([1331])
y shape: torch.Size([])
frame_idx: 0
✓ Frame-level sample checks passed


In [9]:
if len(frame_dataset) >= 5:
    random_indices = random.sample(range(len(frame_dataset)), 5)
else:
    random_indices = list(range(len(frame_dataset)))

print(f"Inspecting frame samples at indices: {random_indices}")

for idx in random_indices:
    sample = frame_dataset[idx]
    print(f"  Index {idx}: ex_id={sample['ex_id']}, frame_idx={sample['frame_idx']}, y={sample['y'].item():.1f}")
    assert sample['x'].shape == (1331,), f"Shape error at index {idx}"
    assert sample['y'].dim() == 0, f"Scalar error at index {idx}"
    assert 0.0 <= sample['y'].item() <= 1.0, f"Binary label error at index {idx}: {sample['y'].item()}"

print("✓ Frame sample inspection passed")

Inspecting frame samples at indices: [141370, 5180, 156922, 158243, 270257]
  Index 141370: ex_id=dev_0000255, frame_idx=380, y=1.0
  Index 5180: ex_id=dev_0000011, frame_idx=374, y=1.0
  Index 156922: ex_id=dev_0000280, frame_idx=1143, y=1.0
  Index 158243: ex_id=dev_0000285, frame_idx=504, y=0.0
  Index 270257: ex_id=dev_0000477, frame_idx=1018, y=0.0
✓ Frame sample inspection passed


In [10]:
finite_count = 0
binary_count = 0
total_checked = min(100, len(frame_dataset))

for i in range(total_checked):
    sample = frame_dataset[i]
    x = sample['x']
    y = sample['y']

    if torch.isfinite(x).all():
        finite_count += 1

    if abs(y.item() - 0.0) < 1e-6 or abs(y.item() - 1.0) < 1e-6:
        binary_count += 1

print(f"Checked {total_checked} samples:")
print(f"  Finite x values: {finite_count}/{total_checked}")
print(f"  Binary y values: {binary_count}/{total_checked}")

assert finite_count == total_checked, f"Found non-finite x values in {total_checked - finite_count} samples"
assert binary_count == total_checked, f"Found non-binary y values in {total_checked - binary_count} samples"
print("✓ All checked frame samples have finite features and binary labels")

Checked 100 samples:
  Finite x values: 100/100
  Binary y values: 100/100
✓ All checked frame samples have finite features and binary labels


## Summary

- The sequence dataset loads cleanly and preserves frame alignment
- The frame dataset wraps the sequence data with context stacking and frame indexing
- Sequence features should be 121-dimensional, frame-stacked features should be 1331-dimensional
- Labels remain binary and feature values remain finite
- Shared setup is defined once, with no duplicated initialization blocks